In [ ]:
import pandas as pd
import numpy as np 
from sklearn.metrics import mean_absolute_error
from typing import List, Dict
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import itertools 
import math
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn import linear_model
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.utils import shuffle
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, StratifiedKFold, GridSearchCV, cross_validate, KFold
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, mean_absolute_error, make_scorer, make_scorer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
from tqdm.notebook import tqdm
from sklearn.model_selection import GridSearchCV
from tqdm.notebook import tqdm_notebook
from sklearn.utils import parallel_backend
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from numpy.random import RandomState
from sklearn.metrics import make_scorer, accuracy_score

In [ ]:
DRN_cell_12_char = pd.read_csv('Новая папка\DRN-serotonin_Female_spikes\CELL 12 DEEP 4850_spikeChar.csv')
DRN_cell_13_char = pd.read_csv('Новая папка\DRN-serotonin_Female_spikes\CELL 13 DEEP 4983_spikeChar.csv')
DRN_cell_14_char = pd.read_csv('Новая папка\DRN-serotonin_Female_spikes\CELL 14 DEEP 6318_spikeChar.csv')
DRN_cell_15_char = pd.read_csv('Новая папка\DRN-serotonin_Female_spikes\CELL 15 DEEP 6426_spikeChar.csv')
DRN_cell_16_char = pd.read_csv('Новая папка\DRN-serotonin_Female_spikes\CELL 16 DEEP 6658_spikeChar.csv')

In [ ]:
VTA_cell_11_char = pd.read_csv('Новая папка\VTA-Dopamine_Female_spikes\VTA cell11 - deep7678_spikeChar.csv')
VTA_cell_12_char = pd.read_csv('Новая папка\VTA-Dopamine_Female_spikes\VTA cell12 - deep7927_spikeChar.csv')
VTA_cell_13_char = pd.read_csv('Новая папка\VTA-Dopamine_Female_spikes\VTA cell13 - deep7957_spikeChar.csv')
VTA_cell_14_char = pd.read_csv('Новая папка\VTA-Dopamine_Female_spikes\VTA cell14 - deep8035_spikeChar.csv')
VTA_cell_15_char = pd.read_csv('Новая папка\VTA-Dopamine_Female_spikes\VTA cell15 - deep8054_spikeChar.csv')
VTA_cell_16_char = pd.read_csv('Новая папка\VTA-Dopamine_Female_spikes\VTA cell16 - deep8202_spikeChar.csv')

In [ ]:
result_DRN = pd.concat([DRN_cell_12_char, DRN_cell_13_char, DRN_cell_14_char, DRN_cell_15_char, DRN_cell_16_char], axis=0) 
result_DRN = result_DRN.sample(frac=1).reset_index(drop=True)
result_DRN


In [ ]:
result_VTA = pd.concat([VTA_cell_11_char, VTA_cell_12_char, VTA_cell_13_char, VTA_cell_14_char, VTA_cell_15_char, VTA_cell_16_char], axis=0) 
result_VTA = result_VTA.sample(frac=1).reset_index(drop=True)
result_VTA

In [ ]:
result_DRN_ML = result_DRN
result_DRN_ML['Neuron_type'] = 0
result_VTA_ML = result_VTA
result_VTA_ML['Neuron_type'] = 1


In [ ]:
result_DRN_ML.to_csv('pivotal DRN table for ML', index=False)
result_VTA_ML.to_csv('pivotal VTA table for ML', index=False)

In [ ]:
def information(data):
    print('Общая информация:')
    display(data.info())
    print('-------------------------------------------------------------------------------------------------')
    print('Пропуски:')
    print(data.isna().sum())
    print('-------------------------------------------------------------------------------------------------')
    print('Описание данных:')
    print(data.describe())
    print('-------------------------------------------------------------------------------------------------')
    print('Дубликаты:')
    print(data.duplicated().sum())
    print('-------------------------------------------------------------------------------------------------')
    print('Ну и графики:')
    print(data.hist(figsize=(15, 10)))
information(result_DRN_ML)

In [ ]:
information(result_VTA_ML)

In [ ]:
result_VTA_ML


In [ ]:
result_DRN_ML_droped = result_DRN_ML.drop(columns=['spike_id','id_clust', 'F1', 'F3', 'F6', 'F8', 'F11', 'F13', 'F15', 'F17', 'F18'], axis=1)
result_VTA_ML_droped = result_VTA_ML.drop(columns=['spike_id','id_clust', 'F1', 'F3', 'F6', 'F8', 'F11', 'F13', 'F15', 'F17', 'F18'], axis=1)

In [ ]:
DRN_VTA_ML = pd.concat([result_DRN_ML_droped, result_VTA_ML_droped], axis=0)

In [ ]:
DRN_VTA_ML_Sampled = DRN_VTA_ML.sample(frac=1).reset_index(drop=True)

In [ ]:
DRN_VTA_ML_Sampled
features = DRN_VTA_ML_Sampled.drop(columns=['Neuron_type'], axis=1)
target = DRN_VTA_ML_Sampled['Neuron_type']

In [ ]:
features_train, features_valid, target_train, target_valid = train_test_split(features, target, random_state=12345, test_size=0.3)

In [ ]:
state = RandomState(12345)

pipe = Pipeline([
    ('imp', SimpleImputer(missing_values=np.nan)),
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=state))
])

params = [
    {
        'imp__strategy': ['mean', 'median'],
        'model': [GradientBoostingClassifier()],
        'model__n_estimators': [50, 100, 200],
        'model__learning_rate': [0.01, 0.05, 0.1],
        'model__max_depth': [3, 4, 5],
        'model__subsample': [0.8, 0.9, 1.0],
    },
    {
        'imp__strategy': ['mean', 'median'],
        'model': [LogisticRegression(penalty='elasticnet', solver='saga')],
        'model__C': [0.001, 0.01, 0.1, 1, 10],  
        'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],  
    },
    
]

In [ ]:
accuracy_scorer = make_scorer(accuracy_score)
cv = KFold(n_splits=5, shuffle=False)
grid_neurons = GridSearchCV(pipe, param_grid=params, cv=cv, n_jobs=-1, scoring=accuracy_scorer)

In [ ]:
grid_neurons.fit(features_train, target_train)

In [ ]:
print('Best Parameters:', grid_neurons.best_params_)
print('Best accuracy Score:', grid_neurons.best_score_)

In [ ]:
best_model = grid_neurons.best_estimator_


In [ ]:
best_model.fit(features_train, target_train)


In [ ]:
train_predictions = grid_neurons.predict(features_train)
valid_predictions = grid_neurons.predict(features_valid)

train_accuracy = accuracy_score(target_train, train_predictions)
valid_accuracy = accuracy_score(target_valid, valid_predictions)

print(f"Accuracy on training set: {train_accuracy:.4f}")
print(f"Accuracy on validation set: {valid_accuracy:.4f}")


In [ ]:
features

In [ ]:
DRN_VTA_ML_Sampled

In [ ]:
(DRN_VTA_ML['Neuron_type'] == 0).sum() / DRN_VTA_ML['Neuron_type'].shape[0]

In [ ]:
(DRN_VTA_ML['Neuron_type'] == 1).sum()/DRN_VTA_ML['Neuron_type'].shape[0]